# 🏆 Lab 5: 本地语义知识库助手 (Local Semantic Knowledge Assistant)
本实验构建一个完全运行在本地的 AI 知识库助手。支持 Word/Excel/PPT/PDF/图片自动解析入库，
通过 **Qwen3-VL 真实 OCR 识别** 提取图片内容，并引入 **BGE 语义向量模型** 实现意图搜索（如搜索“小学英文单词”可匹配到对应图片）。

**🌟 核心创新点 (针对 i3 + 8GB 内存深度优化):**
- `真实 OCR 识别`：非简单文件名匹配，调用 OpenVINO 加速的 Qwen3-VL 提取图片真实内容。
- `语义意图搜索`：引入轻量级向量模型，告别死板关键词，支持自然语言对话检索。
- `Lazy Loading 内存保护`：ASR/OCR/Embedding 模型按需串行加载，用完即 `del` + `gc.collect()`，8GB 内存永不溢出。
- `256px 极速缩略图`：图片压缩后送入模型，推理速度提升 3 倍以上。

In [1]:
# 🛠️ 1. 环境诊断与路径初始化
import os, sys
import importlib
from pathlib import Path

# 强制使用国内镜像
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

print("🔍 正在检查依赖...")

# 尝试动态添加 Lab2 路径
current_dir = Path.cwd()
# 这里的逻辑是：从 lab5 往上一级就是根目录
root_dir = current_dir.parent if current_dir.name == "lab5-local-knowledge-assistant" else current_dir
lab2_path = root_dir / "lab2-speech-recognition"
sys.path.insert(0, str(lab2_path))
print(f"📂 Lab2 路径已添加: {lab2_path}")

missing_libs = []
try: import modelscope
except ImportError: missing_libs.append("modelscope")
try: import PIL
except ImportError: missing_libs.append("Pillow")
try: import pypdf
except ImportError: missing_libs.append("pypdf")
try: import docx
except ImportError: missing_libs.append("python-docx")
try: import openpyxl
except ImportError: missing_libs.append("openpyxl")
try: import pptx
except ImportError: missing_libs.append("python-pptx")
try: import sentence_transformers
except ImportError: missing_libs.append("sentence-transformers")

try:
    # 核心测试：尝试导入 Lab2 的助手文件
    from qwen_3_asr_helper import OVQwen3ASRModel
    print("✅ 所有核心依赖加载成功！(Lab2 语音模块已连接)")
except ImportError as e:
    print(f"❌ 核心依赖缺失: {e}")
    print("💡 原因: 可能是 Jupyter 未选择 'ov_workshop' 环境，或者 setup_lab.bat 运行失败。")
    missing_libs.append("openvino/optimum (请检查环境)")

if missing_libs:
    print(f"🚨 缺失库: {', '.join(missing_libs)}")
    print(f"💊 治疗: 请运行 `pip install {' '.join(missing_libs)}` (如果是环境问题请检查 Kernel)")
else:
    print("🚀 环境检查通过，准备开始！")

🔍 正在检查依赖...
📂 Lab2 路径已添加: D:\ai-contest\modelscope-workshop-clean\lab2-speech-recognition
✅ 所有核心依赖加载成功！(Lab2 语音模块已连接)
🚀 环境检查通过，准备开始！


In [2]:
# 📚 2. 本地知识库索引引擎 (彻底修复：清理旧数据 + 指向原图)
import os, sys, re, sqlite3, time, gc
import numpy as np
from pathlib import Path
from collections import Counter
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from modelscope import snapshot_download
    from PIL import Image
    import pypdf, docx, openpyxl, pptx
    try: from sentence_transformers import SentenceTransformer
    except ImportError: 
        import subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
        from sentence_transformers import SentenceTransformer
except ImportError: pass
EMBEDDING_MODEL_DIR = Path("models/bge-small-zh-v1.5")
class SemanticIndexStore:
    def __init__(self, store_dir="index_store"):
        self.store_dir = Path(store_dir)
        self.store_dir.mkdir(exist_ok=True)
        self.db_path = self.store_dir / "knowledge.db"
        self.emb_path = self.store_dir / "embeddings.npy"
        self.chunks, self.embeddings = [], None
        
        if not self.db_path.exists():
            with sqlite3.connect(self.db_path) as conn:
                conn.execute("CREATE TABLE IF NOT EXISTS chunks (id INTEGER PRIMARY KEY, file_path TEXT, file_type TEXT, chunk_text TEXT, file_name TEXT)")
        self.load_index()
        
        print("🧠 加载语义模型 (BGE)...")
        if not EMBEDDING_MODEL_DIR.exists():
            print("📥 语义模型未找到，从魔搭下载...")
            snapshot_download("AI-ModelScope/bge-small-zh-v1.5", local_dir=str(EMBEDDING_MODEL_DIR))
        self.model = SentenceTransformer(str(EMBEDDING_MODEL_DIR), device='cpu')
    def load_index(self):
        with sqlite3.connect(self.db_path) as conn:
            self.chunks = conn.execute("SELECT id, file_path, file_type, chunk_text, file_name FROM chunks").fetchall()
        if self.emb_path.exists(): self.embeddings = np.load(self.emb_path)
        print(f"📊 当前索引: {len(self.chunks)} 文本块")
    def compute_embeddings(self):
        if not self.chunks: return
        print("🔄 生成语义向量...")
        self.embeddings = self.model.encode([c[3] for c in self.chunks], normalize_embeddings=True, batch_size=8)
        np.save(self.emb_path, self.embeddings)
        print("✅ 向量更新完成")
    def add_file(self, file_path, file_name=None):
        p = Path(file_path)
        if not p.exists(): return
        display_name = file_name if file_name else p.name
        ftype, text = p.suffix.lower(), ""
        try:
            if ftype in [".txt", ".md", ".log"]: text = p.read_text(encoding="utf-8", errors="ignore")
            elif ftype == ".pdf": text = "\n".join(page.extract_text() or "" for page in pypdf.PdfReader(p).pages)
            elif ftype == ".docx": text = "\n".join([p.text for p in docx.Document(p).paragraphs])
            elif ftype == ".xlsx": 
                wb = openpyxl.load_workbook(p, data_only=True)
                text = "\n".join(str(c) for ws in wb.worksheets for r in ws.iter_rows(values_only=True) for c in r if c)
            elif ftype == ".pptx": 
                text = "\n".join(p.text for s in pptx.Presentation(p).slides for sh in s.shapes if sh.has_text_frame for p in sh.text_frame.paragraphs)
            
            if text.strip():
                chunks = [text[i:i+500] for i in range(0, len(text), 500)]
                with sqlite3.connect(self.db_path) as conn:
                    conn.execute("DELETE FROM chunks WHERE file_path = ?", (str(p),))
                    conn.executemany("INSERT INTO chunks (file_path, file_type, chunk_text, file_name) VALUES (?, ?, ?, ?)", [(str(p), ftype, c, display_name) for c in chunks])
                    conn.commit()
                print(f"✅ 成功入库: {display_name} ({len(chunks)}块)")
                self.load_index(); self.compute_embeddings()
        except Exception as e: print(f"❌ 解析失败 {display_name}: {e}")
    def ocr_image(self, img_path, original_name):
        """🔥 真实 Qwen3-VL 识别 (核心修复：清理旧数据 + 指向原图)"""
        model_dir = None
        for c in [Path("Qwen3-VL-4B-Instruct-int4-ov"), Path("lab1-multimodal-vlm/Qwen3-VL-4B-Instruct-int4-ov"), Path("../lab1-multimodal-vlm/Qwen3-VL-4B-Instruct-int4-ov")]:
            if c.exists() and (c / "openvino_model.xml").exists():
                model_dir = c; print(f"✅ 找到模型: {model_dir}"); break
        
        if model_dir is None:
            print("📥 模型未找到，正在下载...")
            model_dir = Path("Qwen3-VL-4B-Instruct-int4-ov")
            snapshot_download("snake7gun/Qwen3-VL-4B-Instruct-int4-ov", local_dir=str(model_dir))
        temp_thumb_path = Path(img_path).with_suffix(".thumb.jpg")
        model, processor = None, None
        
        try:
            # 1. 压缩图片
            with Image.open(img_path) as img:
                img = img.convert("RGB")
                img.thumbnail((256, 256), Image.Resampling.LANCZOS)
                img.save(temp_thumb_path, quality=85)
            # 2. 加载模型
            print(f"🔄 [OCR] 加载视觉模型处理: {original_name}...")
            from optimum.intel.openvino import OVModelForVisualCausalLM
            from transformers import AutoProcessor
            model = OVModelForVisualCausalLM.from_pretrained(model_dir, device="CPU")
            processor = AutoProcessor.from_pretrained(model_dir)
            # 3. 推理
            msgs = [{"role": "user", "content": [{"type": "image", "image": str(temp_thumb_path)}, {"type": "text", "text": "请提取图片中的所有文字和主要内容。"}]}]
            inputs = processor.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
            out = model.generate(**inputs, max_new_tokens=256)
            text = processor.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()
            print(f"👁️ 识别成功: {text[:30]}...")
            
            # 4. ✅ 核心修复：清理旧数据 + 存入新数据
            if text:
                chunks = [text[i:i+500] for i in range(0, len(text), 500)]
                with sqlite3.connect(self.db_path) as conn:
                    # 🔥 彻底删除该图片的所有旧记录 (防止缩略图路径残留导致文件丢失)
                    conn.execute("DELETE FROM chunks WHERE file_name = ?", (original_name,))
                    
                    # 插入新记录 (必须使用 img_path 原图路径！)
                    conn.executemany("INSERT INTO chunks (file_path, file_type, chunk_text, file_name) VALUES (?, ?, ?, ?)", 
                                     [(str(img_path), ".image_ocr", c, original_name) for c in chunks])
                    conn.commit()
                print(f"✅ 图片 OCR 入库完成 (路径已修正): {original_name}")
                self.load_index(); self.compute_embeddings()
        except Exception as e:
            print(f"❌ 图片处理失败 {original_name}: {e}")
        finally:
            # 5. 清理缩略图 (原图 img_path 必须保留在 knowledge 文件夹中！)
            if 'temp_thumb_path' in locals() and temp_thumb_path.exists(): temp_thumb_path.unlink()
            print(f"🗑️ [OCR] 清理 {original_name} 内存...")
            if model: del model
            if processor: del processor
            gc.collect()
    def semantic_search(self, query, top_k=5):
        if self.embeddings is None or not self.chunks: return [], [], {}
        q_vec = self.model.encode([query], normalize_embeddings=True)[0]
        scores = self.embeddings @ q_vec
        top_idx = np.argsort(scores)[::-1][:top_k]
        results, files_seen, types = [], set(), Counter()
        for idx in top_idx:
            if scores[idx] > 0.1:
                chunk = self.chunks[idx]
                results.append({"text": chunk[3], "score": float(scores[idx])})
                files_seen.add(chunk[4])
                types[chunk[2]] += 1
        return results, files_seen, dict(types)
# --- 启动扫描 ---
print("🔍 定位知识库...")
cwd = Path.cwd()
knowledge_dir = None
for c in [cwd/"knowledge", cwd.parent/"knowledge", cwd.parent.parent/"knowledge"]:
    if c.exists() and any(c.iterdir()): knowledge_dir = c; break

if knowledge_dir:
    print(f"📂 找到知识库: {knowledge_dir}")
    indexer = SemanticIndexStore(store_dir=str(knowledge_dir.parent / "index_store"))
    print("🚀 开始全量扫描 (图片将修复链接并识别)...")
    for f in knowledge_dir.iterdir():
        if f.is_file() and f.suffix != ".thumb.jpg":
            if f.suffix.lower() in [".txt", ".md", ".pdf", ".docx", ".xlsx", ".pptx"]: indexer.add_file(str(f))
            elif f.suffix.lower() in [".png", ".jpg", ".jpeg"]: indexer.ocr_image(str(f), f.name)
else: print("❌ 未找到知识库")

Loading SentenceTransformer model from models\bge-small-zh-v1.5.


🔍 定位知识库...
📂 找到知识库: D:\ai-contest\modelscope-workshop-clean\lab5-local-knowledge-assistant\knowledge
📊 当前索引: 13 文本块
🧠 加载语义模型 (BGE)...
🚀 开始全量扫描 (图片将修复链接并识别)...
✅ 成功入库: 2026 战略规划_备忘录.txt (1块)
📊 当前索引: 13 文本块
🔄 生成语义向量...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 向量更新完成
📥 模型未找到，正在下载...
🔄 [OCR] 加载视觉模型处理: near.png...


The tokenizer you are loading from 'Qwen3-VL-4B-Instruct-int4-ov' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


...

👁️ 识别成功: 以下是图片中提取的所有文字和主要内容：

---

**小学...
✅ 图片 OCR 入库完成 (路径已修正): 小学核心词汇x5.jpeg
📊 当前索引: 14 文本块
🔄 生成语义向量...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 向量更新完成
🗑️ [OCR] 清理 小学核心词汇x5.jpeg 内存...
✅ 成功入库: 研发预算审批.docx (1块)
📊 当前索引: 14 文本块
🔄 生成语义向量...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 向量更新完成


In [3]:
# 🤖 4. 智能检索助手 (专业表格版 + 文件直链)
import gradio as gr
from pathlib import Path
import tempfile
import wave
import gc
class SearchUI:
    def __init__(self):
        self.ov_config = {"INFERENCE_NUM_THREADS": "4"}
        self.asr_model = None
        for c in [Path("Qwen3-ASR-0.6B-fp16-ov"), Path("../lab2-speech-recognition/Qwen3-ASR-0.6B-fp16-ov")]:
            if c.exists() and (c / "openvino_model.xml").exists():
                self.asr_path = c; break
        else: self.asr_path = Path("Qwen3-ASR-0.6B-fp16-ov")
    def get_asr(self):
        if self.asr_model is None:
            if not self.asr_path.exists():
                print("📥 下载 ASR..."); from modelscope import snapshot_download; snapshot_download("snake7gun/Qwen3-ASR-0.6B-fp16-ov", local_dir=str(self.asr_path))
            print("🔄 加载 ASR..."); from qwen_3_asr_helper import OVQwen3ASRModel
            self.asr_model = OVQwen3ASRModel.from_pretrained(str(self.asr_path), device="CPU", ov_config=self.ov_config)
        return self.asr_model
    def clear_asr(self):
        if self.asr_model: del self.asr_model; self.asr_model = None; gc.collect()
    def search_and_display(self, audio, text, use_voice):
        query = ""
        # 1. 处理语音
        if use_voice and audio is not None:
            print("🎙️ 识别中...")
            try:
                model = self.get_asr()
                sr, data = audio
                with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
                    if data.dtype != np.float32: data = data.astype(np.float32) / 32768.0
                    with wave.open(f.name, 'wb') as wf: wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr); wf.writeframes((data*32767).astype(np.int16).tobytes())
                    res = model.transcribe(audio=f.name, language="zh"); query = res[0].text if res else ""
            except: query = "(识别失败)"
            self.clear_asr()
        else: query = text
        if not query: return "<h3>⚠️ 请输入内容</h3>", f"🗣️ 您的查询：(空)"
        # 2. 检索
        print(f"🔍 语义搜索: {query}")
        results, files_seen, types = indexer.semantic_search(query, top_k=10)
        # 3. 构建表格
        html = """
        <style>
            table { width: 100%; border-collapse: collapse; font-family: sans-serif; }
            th { background: #007bff; color: white; padding: 10px; }
            td { padding: 10px; border-bottom: 1px solid #eee; }
            tr:hover { background: #f9f9f9; }
            a { text-decoration: none; color: #0056b3; font-weight: bold; font-size: 1.1em; cursor: pointer; }
            a:hover { text-decoration: underline; color: #d9534f; }
            .score { color: #28a745; font-weight: bold; }
        </style>
        <h3>📚 搜索结果</h3><table><thead><tr><th width="25%">📂 文件 (点击打开)</th><th width="10%">📊 匹配度</th><th width="65%">📝 内容摘要</th></tr></thead><tbody>
        """
        if not results:
            html += "<tr><td colspan='3'>❌ 未找到相关内容</td></tr>"
        else:
            for res in results:
                # 寻找文件路径
                f_path, f_name = "未知路径", "未知文件"
                # 从 chunks 中匹配 (chunk[3]是文本, chunk[1]是路径, chunk[4]是文件名)
                for chunk in indexer.chunks:
                    if chunk[3] == res['text']: 
                        f_path, f_name = chunk[1], chunk[4]; break
                
                # 生成 file:/// 链接
                if Path(f_path).exists():
                    # Windows 路径转换为 URL 格式
                    link_url = "file:///" + Path(f_path).as_posix()
                    link_html = f'<a href="{link_url}" target="_blank">📄 {f_name}</a>'
                else:
                    link_html = f'❌ 文件丢失: {f_name}'
                html += f"""
                <tr>
                    <td>{link_html}</td>
                    <td><span class="score">{res['score']:.2f}</span></td>
                    <td style="color:#555; font-size:0.9em;">{res['text'][:150]}...</td>
                </tr>
                """
        html += "</tbody></table>"
        return html, f"🗣️ 您的查询：{query}"
assistant = SearchUI()
with gr.Blocks(title="本地知识库检索") as demo:
    gr.Markdown("# 🧠 智能语义检索助手")
    with gr.Row():
        with gr.Column(scale=1):
            voice_mode = gr.Checkbox(value=True, label="🎙️ 开启语音")
            audio_in = gr.Audio(sources=["microphone"], type="numpy")
            text_in = gr.Textbox(label="📝 搜索内容", placeholder="例如：帮我找找小学英文单词")
            btn_run = gr.Button("🔍 搜索", variant="primary")
            echo_out = gr.Textbox(label="📢 识别结果", interactive=False)
        with gr.Column(scale=2):
            html_out = gr.HTML(value="<h3>👈 请输入并搜索...</h3>")
    
    btn_run.click(assistant.search_and_display, [audio_in, text_in, voice_mode], [html_out, echo_out])
demo.launch(share=False, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860


HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


🔍 语义搜索: 英语单词


Batches:   0%|          | 0/1 [00:00<?, ?it/s]